# 25280091 — Extra Credit (PA4)

QA as subgraph: `syntax_checker` → `logic_tester` → `performance_auditor` (single parent node).


In [ ]:
%pip install -U typing_extensions langgraph langchain databricks-langchain pygame
dbutils.library.restartPython()


In [ ]:
import os
import operator
import subprocess

from typing import Annotated, List

from typing_extensions import TypedDict
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from databricks_langchain import ChatDatabricks
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# ChatDatabricks uses workspace auth. Optional cluster env DATABRICKS_CHAT_ENDPOINT for endpoint name.
_PA4_RUN_LOG_PREFIX = "__PA4_RUN_LOG__:\n"

MOSAIC_CHAT_ENDPOINT = (
    (os.environ.get("DATABRICKS_CHAT_ENDPOINT") or "").strip()
    or "databricks-meta-llama-3.1-405b-instruct"
)


def make_chat_model(
    model_env: str, temp_env: str, default_model: str, default_temp: float
) -> ChatDatabricks:
    _, _ = model_env, default_model
    temp = float(os.getenv(temp_env, str(default_temp)))
    return ChatDatabricks(endpoint=MOSAIC_CHAT_ENDPOINT, temperature=temp)


print("ChatDatabricks endpoint:", MOSAIC_CHAT_ENDPOINT)


def architect_spec_from_state(state: dict) -> str:
    parts: List[str] = []
    for m in state.get("architect_messages") or []:
        if isinstance(m, AIMessage):
            parts.append(str(m.content))
    return "\n\n".join(parts).strip()


class GameState(TypedDict):
    """Same eight keys as PA4 base notebook."""

    director_messages: Annotated[List[BaseMessage], add_messages]
    architect_messages: Annotated[List[BaseMessage], add_messages]
    engineer_code: str
    qa_feedback: str
    current_actor: str
    iteration: int
    iteration_score: Annotated[List[float], operator.add]
    file_saved: bool


class QAState(TypedDict):
    run_output: str
    design_requirements: str
    engineer_code: str
    syntax_report: str
    logic_report: str
    qa_feedback: str
    score: float


def syntax_checker_node(state: QAState):
    code = (state.get("engineer_code") or "").strip()
    run_output = (state.get("run_output") or "").strip()

    lines: List[str] = []
    if not code:
        lines.append("Engineer code is empty.")
    else:
        try:
            compile(code, "<engineer_code>", "exec")
            lines.append("Python syntax compile check: PASS")
        except Exception as exc:
            lines.append(f"Python syntax compile check: FAIL -> {exc}")

    if run_output:
        lines.append("Execution evidence:\n" + run_output[:4000])

    report = "\n".join(lines).strip()
    return {"syntax_report": report}


def logic_tester_node(state: QAState):
    from langchain_core.messages import SystemMessage

    design_requirements = (state.get("design_requirements") or "").strip()
    engineer_code = (state.get("engineer_code") or "").strip()
    syntax_report = (state.get("syntax_report") or "").strip()

    sys_prompt = """You are QA logic tester for PA4 pygame.
Compare behavior to requirements. Numbered findings; end with Suggested fixes: bullets."""

    body = f"""## Requirements
{design_requirements[:12000]}

## Syntax checker
{syntax_report[:6000]}

## Code
{engineer_code[:30000]}
"""

    llm = make_chat_model(
        model_env="QA_MODEL",
        temp_env="QA_TEMP",
        default_model="",
        default_temp=0.15,
    )
    ai = llm.invoke([SystemMessage(content=sys_prompt), HumanMessage(content=body)])
    logic_report = (ai.content if hasattr(ai, "content") else str(ai)).strip()
    return {"logic_report": logic_report}


def performance_auditor_node(state: QAState):
    import re

    from langchain_core.messages import SystemMessage

    syntax_report = (state.get("syntax_report") or "").strip()
    logic_report = (state.get("logic_report") or "").strip()

    consolidated = (
        "QA subgraph report\n\n"
        f"[Syntax Checker]\n{syntax_report}\n\n"
        f"[Logic Tester]\n{logic_report}"
    ).strip()

    judge = make_chat_model(
        model_env="SCORE_MODEL",
        temp_env="SCORE_TEMP",
        default_model="",
        default_temp=0.0,
    )
    ai = judge.invoke(
        [
            SystemMessage(
                content="Return ONLY one decimal from 1.0 to 10.0 judging PA4 pygame readiness."
            ),
            HumanMessage(content=consolidated[:16000]),
        ]
    )
    text = (ai.content if hasattr(ai, "content") else str(ai)).strip()
    m = re.search(r"[-+]?\d*\.?\d+", text)
    score = float(m.group(0)) if m else 0.0
    score = max(1.0, min(10.0, score))

    return {
        "qa_feedback": consolidated,
        "score": score,
    }


qa_builder = StateGraph(QAState)
qa_builder.add_node("syntax_checker", syntax_checker_node)
qa_builder.add_node("logic_tester", logic_tester_node)
qa_builder.add_node("performance_auditor", performance_auditor_node)
qa_builder.add_edge(START, "syntax_checker")
qa_builder.add_edge("syntax_checker", "logic_tester")
qa_builder.add_edge("logic_tester", "performance_auditor")
qa_builder.add_edge("performance_auditor", END)
qa_subgraph = qa_builder.compile()


def _run_payload_from_feedback(state: GameState) -> str:
    q = (state.get("qa_feedback") or "").strip()
    if q.startswith(_PA4_RUN_LOG_PREFIX):
        return q[len(_PA4_RUN_LOG_PREFIX) :].strip()
    return q.strip()


def qa_subgraph_node(state: GameState):
    qa_in: QAState = {
        "run_output": _run_payload_from_feedback(state),
        "design_requirements": architect_spec_from_state(state),
        "engineer_code": state.get("engineer_code") or "",
        "syntax_report": "",
        "logic_report": "",
        "qa_feedback": "",
        "score": 0.0,
    }
    qa_out = qa_subgraph.invoke(qa_in)
    score = float(qa_out.get("score", 0.0))
    prev_it = int(state.get("iteration", 0))
    return {
        "qa_feedback": qa_out.get("qa_feedback") or "",
        "iteration_score": [score],
        "iteration": prev_it + 1,
        "current_actor": "qa_subgraph",
    }


def director_node(state: GameState):
    director_msg = input("Awaiting Director Prompt: ").strip()
    return {
        "director_messages": [
            HumanMessage(
                content=director_msg,
                additional_kwargs={"role": "director"},
            )
        ],
        "current_actor": "director",
    }


def architect_node(state: GameState):
    it = int(state.get("iteration", 0))
    print(f"\n========== ITERATION {it} (architect pass) ==========")

    brief = ""
    for m in reversed(state.get("director_messages") or []):
        if isinstance(m, HumanMessage) and m.additional_kwargs.get("role") == "director":
            brief = str(m.content)
            break

    qa_fb = (state.get("qa_feedback") or "").strip()
    if it > 0 and qa_fb and not qa_fb.startswith(_PA4_RUN_LOG_PREFIX):
        print("[QA feedback from last run]\n", qa_fb, sep="")

    print(
        "Architect: PA4 pygame Dino Runner (flying + ground obstacles, jump/duck, high score).\n"
        f"Director brief: {brief!r}\n"
    )
    spec = input("Architect requirements (multi-line; end empty line):\n")
    while True:
        line = input()
        if not line.strip():
            break
        spec += "\n" + line

    spec = spec.strip()
    if not spec and brief:
        spec = brief

    return {
        "architect_messages": [AIMessage(content=f"[Architect spec]\n{spec}")],
        "current_actor": "architect",
    }


def engineer_node(state: GameState):
    from langchain_core.messages import SystemMessage

    spec = architect_spec_from_state(state)
    prev_code = (state.get("engineer_code") or "").strip()
    qa_fb = (state.get("qa_feedback") or "").strip()

    sys_prompt = """Implement PA4 Chrome-style DINO RUNNER in one pygame file.

Must include: flying hazards, ground obstacles, jump physics with gravity,
duck key, persisted high score across runs.

Return ONLY Python inside ```python ... ```."""

    chunks = ["### Specification", spec or "(none)", "### QA feedback", qa_fb or "(none yet)"]
    if prev_code:
        chunks.extend(["### Existing code", prev_code[:120_000]])

    human = HumanMessage(content="\n\n".join(chunks))
    llm = make_chat_model(
        model_env="ENGINEER_MODEL",
        temp_env="ENGINEER_TEMP",
        default_model="",
        default_temp=0.2,
    )
    ai = llm.invoke([SystemMessage(content=sys_prompt), human])
    reply = getattr(ai, "content", str(ai)).strip()

    def extract_py(src: str) -> str:
        fence = "```python"
        if fence in src:
            blob = src.split(fence, 1)[1]
            return blob.split("```", 1)[0].strip()
        if "```" in src:
            return src.split("```", 1)[1].split("```", 1)[0].strip()
        return src.strip()

    new_code = extract_py(reply)
    return {"engineer_code": new_code, "current_actor": "engineer"}


def _runner_filename() -> str:
    roll = (os.getenv("ROLL_NUMBER") or "").strip()
    return f"{roll}_dino_runner.py" if roll else "dino_runner.py"


def file_writer(state: GameState):
    code = (state.get("engineer_code") or "").strip()
    if not code:
        code = "# empty\npass\n"

    fn = _runner_filename()
    path = os.path.abspath(fn)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(code)

    snippet = code if len(code) <= 500 else code[:500] + "\n... [truncated]"
    print(snippet)
    print(f"\n[File saved: {path}]")
    return {"file_saved": True, "current_actor": "file_writer"}


def run_code(state: GameState):
    import sys

    print("\n===== CODE EXECUTION =====")
    choice = input("Run the generated game? (y/n): ").strip().lower()
    fn = _runner_filename()

    if choice != "y":
        log = f"[skipped] User chose not to run {fn}."
        return {
            "qa_feedback": _PA4_RUN_LOG_PREFIX + log,
            "current_actor": "execution_manager",
        }

    script = os.path.abspath(fn)
    exe = sys.executable or "python"
    chunks: List[str] = []

    chk = subprocess.run(
        [exe, "-m", "py_compile", script],
        capture_output=True,
        text=True,
        timeout=120,
        cwd=os.path.dirname(script) or ".",
    )
    chunks.append(f"=== py_compile {fn} exit={chk.returncode} ===")
    if chk.stdout.strip():
        chunks.append(chk.stdout.rstrip())
    if chk.stderr.strip():
        chunks.append(chk.stderr.rstrip())
    if chk.returncode != 0:
        return {
            "qa_feedback": _PA4_RUN_LOG_PREFIX + "\n".join(chunks),
            "current_actor": "execution_manager",
        }

    try:
        proc = subprocess.run(
            [exe, script],
            capture_output=True,
            text=True,
            timeout=30,
            cwd=os.path.dirname(script) or ".",
        )
        chunks.append(f"=== run {fn} exit={proc.returncode} ===")
        if proc.stdout.strip():
            chunks.append(proc.stdout.rstrip())
        if proc.stderr.strip():
            chunks.append(proc.stderr.rstrip())
    except subprocess.TimeoutExpired as exc:
        chunks.append("=== run timeout (pygame interactive) ===")
        if exc.stdout:
            chunks.append(exc.stdout.strip())
        if exc.stderr:
            chunks.append(exc.stderr.strip())

    return {
        "qa_feedback": _PA4_RUN_LOG_PREFIX + "\n".join(chunks),
        "current_actor": "execution_manager",
    }


def human_route_after_qa(state: GameState):
    scores = state.get("iteration_score") or []
    last = float(scores[-1]) if scores else float("nan")
    it = int(state.get("iteration", 0))

    print(f"\n[Human] iteration={it} | score={'%.1f' % last if scores else 'n/a'} / 10")

    ans = input(
        'Refine to engineer? Type "y" to continue; else END: '
    ).strip().lower()
    return "refine" if ans == "y" else "finish"


checkpointer = MemorySaver()
parent = StateGraph(GameState)
parent.add_node("director", director_node)
parent.add_node("architect", architect_node)
parent.add_node("engineer", engineer_node)
parent.add_node("file_writer", file_writer)
parent.add_node("run_code", run_code)
parent.add_node("qa_subgraph", qa_subgraph_node)

parent.add_edge(START, "director")
parent.add_edge("director", "architect")
parent.add_edge("architect", "engineer")
parent.add_edge("engineer", "file_writer")
parent.add_edge("file_writer", "run_code")
parent.add_edge("run_code", "qa_subgraph")
parent.add_conditional_edges(
    "qa_subgraph",
    human_route_after_qa,
    {"refine": "engineer", "finish": END},
)

ec_app = parent.compile(checkpointer=checkpointer)



In [ ]:
from IPython.display import Image, display

# Render as PNG (parent graph; subgraph appears as qa_subgraph node)
display(Image(ec_app.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": os.getenv("EC_THREAD_ID", "ec-thread-1")}}

initial_state: GameState = {
    "director_messages": [],
    "architect_messages": [],
    "engineer_code": "",
    "qa_feedback": "",
    "current_actor": "",
    "iteration": 0,
    "iteration_score": [],
    "file_saved": False,
}

print("--- STARTING EC WORKFLOW ---")
for event in ec_app.stream(initial_state, config=config):
    for node in event.keys():
        print(f"\n===== {node.upper()} =====")

final = ec_app.get_state(config)
vals = final.values
print("\n--- EC DONE ---")
print(f"iteration={vals.get('iteration')!r} iteration_score={vals.get('iteration_score')!r}")
print("\nQA feedback preview:\n", (vals.get("qa_feedback") or "")[:1600])

